# MiniCells Core Validation 002C — Oracle Sparse-Assembly Representation Tomography

Final 002-series diagnostic. This is evaluator-only oracle tomography: it does not edit the model and cannot change the frozen 002 or 002B outcomes.


In [ ]:
from pathlib import Path
import os, subprocess, sys, json, shutil

BRANCH = 'codex/core-validation-002c-oracle-tomography'
REPO = 'https://github.com/ArcheLabs/mini-cells.git'
ROOT = Path('/kaggle/working/mini-cells')
if not ROOT.exists():
    subprocess.run(['git','clone','--branch',BRANCH,'--single-branch',REPO,str(ROOT)], check=True)
else:
    subprocess.run(['git','fetch','origin',BRANCH], cwd=ROOT, check=True)
    subprocess.run(['git','checkout',BRANCH], cwd=ROOT, check=True)
    subprocess.run(['git','reset','--hard',f'origin/{BRANCH}'], cwd=ROOT, check=True)
os.chdir(ROOT)
print('HEAD', subprocess.check_output(['git','rev-parse','HEAD'], text=True).strip())
print('TREE', subprocess.check_output(['git','rev-parse','HEAD^{tree}'], text=True).strip())


In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.[dev]'], check=True)
subprocess.run([sys.executable,'-m','pytest','-q','tests/test_core_validation_002.py','tests/test_core_validation_002b.py','tests/test_core_validation_002c.py'], check=True)
SMOKE_OUT = ROOT / 'results' / 'core-validation-002c-smoke'
if SMOKE_OUT.exists(): shutil.rmtree(SMOKE_OUT)
subprocess.run([sys.executable,'scripts/run_core_validation_002c.py','--smoke','--device','cpu','--out',str(SMOKE_OUT)], check=True)
smoke = json.loads((SMOKE_OUT/'raw.json').read_text())
assert smoke['decision']['status'] == 'SMOKE_ONLY'
assert smoke['decision']['scientific_decision'] is False
print('002C unit tests and CPU smoke passed.')


In [ ]:
import torch
print({'torch':torch.__version__,'cuda':torch.version.cuda,'gpu_count':torch.cuda.device_count(),'gpus':[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]})
assert torch.cuda.is_available(), 'Formal Core Validation 002C requires CUDA.'


In [ ]:
OUT = ROOT / 'results' / 'core-validation-002c-oracle-tomography'
if OUT.exists(): shutil.rmtree(OUT)
subprocess.run([sys.executable,'scripts/run_core_validation_002c.py','--device','cuda'], check=True)
subprocess.run([sys.executable,'scripts/report_core_validation_002c.py'], check=True)
decision = json.loads((OUT/'decision.json').read_text())
print(json.dumps(decision, indent=2, sort_keys=True))


In [ ]:
import pandas as pd
display(pd.read_csv(OUT/'gate-summary.csv'))
display(pd.read_csv(OUT/'seed-summary.csv'))


In [ ]:
PUBLISH = True
if PUBLISH:
    subprocess.run([sys.executable,'scripts/publish_core_validation_002c.py','--push'], check=True)
else:
    subprocess.run([sys.executable,'scripts/publish_core_validation_002c.py'], check=True)
